# National Social Assistance Programme (NSAP) - Model Training
This notebook generates a synthetic dataset based on the official eligibility guidelines of the NSAP schemes (IGNOAPS, IGNWPS, IGNDPS, NFBS, Annapurna), trains a Random Forest Classifier, and evaluates its performance.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
# Configuration
MODEL_PATH = "nsap_model.joblib"
DATA_SIZE = 12000
RANDOM_STATE = 42

In [ ]:
def generate_synthetic_data(num_samples=DATA_SIZE):
    np.random.seed(RANDOM_STATE)
    
    age = np.random.randint(18, 90, size=num_samples)
    gender = np.random.choice([0, 1], size=num_samples, p=[0.5, 0.5])  # 0: Male, 1: Female
    is_bpl = np.random.choice([0, 1], size=num_samples, p=[0.4, 0.6])  # 60% BPL
    disability_percentage = np.random.randint(0, 100, size=num_samples)
    is_widow = np.random.choice([0, 1], size=num_samples, p=[0.8, 0.2])  # 20% Widows
    breadwinner_deceased = np.random.choice([0, 1], size=num_samples, p=[0.9, 0.1])  # 10% deceased breadwinner
    receiving_other_pension = np.random.choice([0, 1], size=num_samples, p=[0.7, 0.3])  # 30% receiving other pension

    is_widow = np.where(gender == 1, is_widow, 0)
    
    labels = []
    for i in range(num_samples):
        if is_bpl[i] == 0:
            labels.append("Ineligible")
            continue
        if 18 <= age[i] <= 79 and disability_percentage[i] >= 80:
            labels.append("IGNDPS")
        elif 18 <= age[i] <= 59 and breadwinner_deceased[i] == 1:
            labels.append("NFBS")
        elif gender[i] == 1 and is_widow[i] == 1 and 40 <= age[i] <= 79:
            labels.append("IGNWPS")
        elif age[i] >= 65 and receiving_other_pension[i] == 0:
            labels.append("Annapurna")
        elif age[i] >= 60:
            labels.append("IGNOAPS")
        else:
            labels.append("Ineligible")

    df = pd.DataFrame({
        'age': age,
        'gender': gender,
        'is_bpl': is_bpl,
        'disability_percentage': disability_percentage,
        'is_widow': is_widow,
        'breadwinner_deceased': breadwinner_deceased,
        'receiving_other_pension': receiving_other_pension,
        'scheme': labels
    })
    return df

In [ ]:
print("--- Generating Synthetic NSAP Dataset ---")
df = generate_synthetic_data()
print(df['scheme'].value_counts())

X = df.drop(columns=['scheme'])
y = df['scheme']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print("\n--- Training Random Forest Multi-Class Classifier ---")
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nSaving model to joblib...")
joblib.dump(model, MODEL_PATH)
print("Model saved successfully!")